# M2 · 物理性能剖析 —— GPU 时间都花在哪里？

**学习目标：** 用 `rocprofv3` 在你的 W7900 上采集 Qwen-Image 工作负载的真实内核轨迹，把 GPU 忙碌时间归因到 GEMM / 注意力 / 归一化 / 逐元素各类，并区分*GPU 忙碌时间*与*端到端延迟*。冻结实验数据中的 61.3% GEMM 占比只是针对你这台主机的假设。

参考讲义：[M02 讲义](../course/M02_profiling/LESSON.zh-CN.md) · [English](./02_profile_and_bottlenecks.ipynb)

In [ ]:
from pathlib import Path
import json
import statistics
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the course repository root (see notebooks/README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
COURSE_MODE = "REFERENCE"  # <- edit once: REFERENCE | GPU_CLASSROOM | GPU_RESEARCH
from lab import notebook_runtime as rt
from lab import notebook_support as ns
session = rt.begin(COURSE_MODE, notebook="02")

def _stamp(kind: str, value: str) -> None:
    """Print a visible origin tag so no number is mistaken for another kind."""
    print(f"[{kind}] {value}")

MEASURED_TODAY = "measured on THIS GPU in THIS session (physical)"
HISTORICAL   = "frozen reference evidence from the committed campaign"
ESTIMATE     = "analytical estimate or simulation - NOT a measurement"

def gpu_cell(fn):
    """Run a physical exercise only in GPU modes; print why otherwise."""
    if session.mode != "REFERENCE":
        return fn()
    print("skipped physical exercise: COURSE_MODE=REFERENCE (no GPU work launched)")
    return None

## 1 · 由运行记录做流水线分解
先做低开销的一层：用 B0 运行记录中的节点计时，把 warm 端到端延迟拆成 KSampler（DiT 去噪）、VAE 解码、文本编码与推导残差。

In [ ]:
def own_pipeline_layer():
    out = session.run_profile(layers="pipeline")
    return out

pipe_dir = gpu_cell(own_pipeline_layer)
if pipe_dir is not None:
    pb = json.loads((pipe_dir / "pipeline_breakdown.json").read_text())
    _stamp(MEASURED_TODAY, f"pipeline layer from YOUR records: {pipe_dir.name}")
else:
    pb = ns.read_json("evidence/results/p2-profile-profile/pipeline_breakdown.json")
    _stamp(HISTORICAL, "frozen campaign pipeline breakdown")
for k in ("n_warm", "T_total_warm_s", "T_dit_denoise_s", "T_vae_decode_s",
          "T_text_encoder_cached_s", "T_other_warm_s", "step_median_s"):
    print(f"  {k:28s} {pb.get(k)}")
print("  method note:", pb.get("method", {}).get(
    "T_other_warm_s", pb.get("note", "derived by subtraction")))

## 2 · 内核层（rocprofv3）
昂贵的一层会在有界 `rocprofv3` 窗口内追踪**真实的 ComfyUI 服务器进程**（DiT 阶段与文本编码阶段）。拦截会引入额外开销，因此**追踪时的墙钟时间永远不是性能结论**——只用于构成分析。课堂上通常*消费*资格认证实验的轨迹；设置 `RUN_KERNEL_TRACE = True`（GPU 模式）可启动你自己的有界窗口（约 10 分钟）。

In [ ]:
RUN_KERNEL_TRACE = False  # opt-in: ~10 min of physical GPU tracing

FROZEN_TRACE = ROOT / "evidence/results/p2-profile-profile/rocprof-dit/analysis.json"

def load_kernel_analysis():
    cands = sorted((ROOT / "runs").glob("*-profile/rocprof-dit/analysis.json"))
    if cands:
        return json.loads(cands[-1].read_text()), MEASURED_TODAY
    if FROZEN_TRACE.is_file():
        return json.loads(FROZEN_TRACE.read_text()), HISTORICAL
    return None, None

loaded = gpu_cell(load_kernel_analysis)
ka = loaded[0] if loaded else None
ka_origin = loaded[1] if loaded else None
if ka is None and RUN_KERNEL_TRACE and session.mode != "REFERENCE":
    session.run_profile(layers="pipeline,kernel")
    loaded = load_kernel_analysis()
    ka, ka_origin = (loaded[0], loaded[1]) if loaded else (None, None)
if ka is not None:
    _stamp(ka_origin, "kernel trace analysis (traced run; composition only)")
    print("mode:", ka.get("mode"), "| kernels:", ka.get("n_kernels"),
          "| gpu_busy_s:", ka.get("gpu_busy_s"), f"({ka.get('gpu_busy_pct')}%)")
    by = ka.get("by_class", {})
    total = sum(v["total_s"] for v in by.values()) or 1.0
    for name, v in sorted(by.items(), key=lambda kv: -kv[1]["total_s"]):
        print(f"  {name:12s} {v['total_s']:8.2f}s  {v['total_s']/total*100:5.1f}%  "
              f"n={v.get('n', v.get('count', '?'))}")
else:
    print("No kernel trace available on this host")

## 3 · 从构成到 Amdahl 上限
若 GEMM 占 GPU 忙碌时间的比例为 `g`，即使 GEMM 无限快，内核层加速上限也只有 1/(1−g)。用**你自己的**轨迹计算这个上限——并记住它只作用于 GPU 忙碌时间，而不是端到端延迟（排队、文本编码、VAE、主机开销都不在其中）。

In [ ]:
if ka is not None:
    by = ka.get("by_class", {})
    gemm = by.get("gemm", {}).get("total_s", 0.0)
    total_busy = sum(v["total_s"] for v in by.values()) or 1.0
    g = gemm / total_busy
    ceiling = 1.0 / (1.0 - g) if g < 1 else float("inf")
    _stamp(ESTIMATE,
           f"Amdahl kernel-layer ceiling {ceiling:.2f}x from YOUR measured "
           f"g={g:.3f} (derived, not measured)")
    print(f"gemm share g = {g*100:.1f}%  ->  ceiling {ceiling:.2f}x")
else:
    g, ceiling = 0.613, 1.0 / (1 - 0.613)
    _stamp(HISTORICAL, f"frozen campaign g=61.3% -> ceiling {ceiling:.2f}x")

## 4 · 动手练习
1. 为什么被追踪的运行端到端更慢，但内核构成分析依然有效？
2. M1 的端到端 warm 中位数远大于 GPU 忙碌总时间——列出造成差距的三个具体因素。
3. 你下一个会优化哪一类？若把它完全消除，它对应的 Amdahl 上限是多少？

## 验收标准

- [ ] 你能指出哪些数字是**今天在这块 GPU 上实测的**、哪些是
      **历史参考值**、哪些是**估计/模拟**。
- [ ] 你产生的每次物理运行都在 `runs/` 下有可见的 `run_id`。
- [ ] 你没有把 smoke/课堂样本冒充为正式协议结果。
- [ ] 你能解释你报告的每项测量的一个局限性。

## 复现与贡献

```bash
python -m lab doctor                                  # 准入证据 -> runs/doctor.json
python -m lab run --experiment B0 --date <your-tag>  # 完整研究协议
python -m lab report --scope student --kind all       # 只汇总你自己的 runs
```

提交你的 `runs/<run-id>/*.json` 记录（绝不写 `evidence/`），并在 PR 中说明：
假设 → 协议 → 数字 → 局限。欢迎负结果——测得的退化是科学结果，不是失败。

```bash
python -m lab profile --layers pipeline,kernel  # physical rocprofv3 trace
```